# ResNet18 — 전처리 1 / 1.5 / 2 비교

**실험 준비 완료, 아직 학습 결과 없음.** 본 학습은 이 노트북을 Colab GPU에서 실행한다.

| 버전 | 학습 사진 | 입력 처리 |
|---|---|---|
| 1 | 현환님 전처리 통과 2,652장 | 기존 고정 크롭 그대로, 마스킹 없음 |
| 1.5 | 버전 1과 정확히 같은 2,652장 | 질감 v2의 `mask_bed`와 동일한 마스킹 추가 |
| 2 | 기존 2,652장＋형상 v2 SUPPORTED 804장 | 기존 사진은 1.5와 동일, 재활용 사진은 저장된 형상 마스크 |

**1↔1.5:** 같은 사진에서 마스킹 효과. **1.5↔2:** 마스킹을 유지하며 재활용 학습 사진 추가 효과.
불확실 148장과 미검출 396장, 합계 544장은 이번 실험에서 제외한다. 면적 10% 같은 새 제외 기준은 추가하지 않는다.

1000×1000 고정 크롭 `x=250:1250, y=0:1000` → 모델 입력 512×512. 작은 영역도 확대하지 않는다.
마스크 안쪽 원본 픽셀은 유지하고 밖만 RGB (124,116,104)로 채운다. 검토용 색 표시 사진은 학습하지 않는다.
질감 마스크가 빈 영역을 반환한 기존 통과 사진은 **원본 유지**하고 `mask_failures.csv`에 기록한다. 전체를 회색으로 지우거나 선별 사진 수를 바꾸지 않는다. 1.5와 2에 동일하게 적용한다.

## 방법과 해석 범위

- ResNet18 ImageNet 사전학습 가중치로 전체 미세조정. seed 0, 기존 시간 블록 5-fold, 30프레임 buffer.
- 같은 시작 가중치·증강·옵티마이저·입력 해상도·평가 사진을 사용한다.
- `ROUNDS=10`: 버전 1의 10 epoch에 가까운 **동일 optimizer 업데이트 예산**. 각 round의 step 수는 버전 1 학습 장수÷batch의 올림이다. 세 버전에 같은 full batch 수를 제공한다. 버전 2는 사진이 많아 사진당 반복 횟수가 적다. 실제 업데이트와 본 사진 수도 저장한다.
- 검증 점수로 조기 종료/체크포인트 선택하지 않는다. 고정 마지막 round만 보고한다.
- 공통 평가: 3,456장의 각자 미학습 fold 예측(OOF). 기존 통과/재활용 후보를 따로 집계한다.
- 버전 1·1.5의 재활용 사진 점수는 **원래 보류할 사진을 강제로 넣은 진단 결과**다. 실제 통과 집합의 점수와 분리한다. 보류는 정상이 아니다.

### 핵심 한계

클래스별 출력 1회라 새 출력 일반화 실험이 아니다. 통과 사진도 100% 클린 정답은 아니다.
재활용 후보는 오프라인 공정표와 같은 촬영 문맥을 사용했다. 새 촬영 자동 동기화 검증은 아니다.
마스크 정답 IoU는 없고, 회색 면적/밝기/공정 위치 지름길도 해결된 것이 아니다.
기존 통과는 질감 마스크, 재활용은 형상 마스크라 두 방법의 차이가 입력에 남을 수 있다.
1.5↔2는 실용적인 데이터 추가 비교이며, 사진 수·공정 유형 효과를 서로 분리하는 실험은 아니다.

## 준비물

Drive `산공종설/`에 `underfill_originals_4000.zip`(기존 파일)과 **`resnet18_threeway_bundle_20261005.zip`**(이번 새 파일)을 둔다.
노트북은 Colab의 **파일 → 노트북 업로드**로 연다. GPU 런타임 선택 후 위에서부터 실행.
처음에는 입력 점검 셀을 보고, 이상 없으면 본 학습 셀부터 이후 셀을 실행한다.
3버전×5fold=15회 학습. L4 기준 학습 약 40~90분 [추정], T4/Drive 속도에 따라 길어질 수 있다.
로그와 매 round 체크포인트를 Drive에 보관한다. 완료한 fold는 건너뛰고 중단한 fold는 마지막 완료 round에서 재개한다.


## 1. GPU 확인과 설정
새 패키지는 설치하지 않는다. Colab 기본 PyTorch/torchvision을 사용하고 실제 버전을 기록한다. 설정을 바꾼 재실험은 `RUN_NAME`도 바꿔야 기존 결과를 보존할 수 있다.

In [ ]:
from pathlib import Path
import os, json, subprocess, sys
import torch, torchvision
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], text=True))
assert torch.cuda.is_available(), "런타임 → 런타임 유형 변경 → GPU를 선택해줘."
print("torch", torch.__version__, "torchvision", torchvision.__version__)
IMG_SIZE = 512
BATCH_SIZE = 16
ROUNDS = 10
SEEDS = [0]
FOLDS = [0, 1, 2, 3, 4]
VARIANTS = ["v1", "v1_5", "v2"]
RUN_NAME = "resnet18_threeway_512_round10_seed0_20261005"
FAST = True  # 빠른 예비 실험. 같은 seed라도 미세한 수치 차이는 가능.


## 2. Drive 연결
아래 경로에서 두 ZIP 파일을 확인한다. 새 묶음에는 목록·마스크·코드만 있어서 원본 사진을 중복 업로드하지 않는다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/산공종설")
ORIGINAL_ZIP = DRIVE_DIR / "underfill_originals_4000.zip"
BUNDLE_ZIP = DRIVE_DIR / "resnet18_threeway_bundle_20261005.zip"
RESULTS = DRIVE_DIR / "results" / RUN_NAME
for p in (ORIGINAL_ZIP, BUNDLE_ZIP):
    assert p.is_file(), f"Drive에 파일이 없어: {p}"
    print(p.name, round(p.stat().st_size / 1024**2, 1), "MiB")
RESULTS.mkdir(parents=True, exist_ok=True)


## 3. 로컬 SSD에 압축 해제·목록 확인
Drive에서 사진을 매 batch 읽지 않는다. 경로와 코드/마스크 해시가 맞지 않으면 실행을 멈춘다.

In [ ]:
import csv, hashlib, shutil, zipfile
WORK = Path("/content/resnet18_threeway_20261005")
WORK.mkdir(exist_ok=True)
BUNDLE = WORK / "bundle"
ORIGINALS = WORK / "originals"

def extract_checked(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for info in z.infolist():
            target = (destination / info.filename.replace("\\", "/")).resolve()
            assert target.is_relative_to(destination.resolve()), "잘못된 ZIP 경로"
        z.extractall(destination)

for source, destination in ((BUNDLE_ZIP, BUNDLE), (ORIGINAL_ZIP, ORIGINALS)):
    local_zip = WORK / source.name
    if not local_zip.exists():
        shutil.copy2(source, local_zip)
    extract_checked(local_zip, destination)
checksums = json.loads((BUNDLE / "bundle_checksums.json").read_text())
for name, expected in checksums.items():
    actual = hashlib.sha256((BUNDLE / name).read_bytes()).hexdigest()
    assert actual == expected, f"파일 내용이 달라짐: {name}"
with (BUNDLE / "manifest.csv").open(encoding="utf-8-sig") as f:
    manifest = list(csv.DictReader(f))
assert len(manifest) == 4000
for row in manifest:
    assert (ORIGINALS / row["source"]).is_file(), row["source"]
print("원본 경로 4,000장 확인, 코드/마스크 해시 확인")
import pandas as pd
display(pd.read_csv(BUNDLE / "coverage.csv"))
display(pd.read_csv(BUNDLE / "split_summary.csv").head(6))
preparation = json.loads((BUNDLE / "preparation.json").read_text(encoding="utf-8"))
print("질감 빈 마스크 → 원본 유지:", preparation["primary_mask_empty_fallback_count"], "장")
if (BUNDLE / "mask_failures.csv").exists():
    display(pd.read_csv(BUNDLE / "mask_failures.csv")[["source", "mask_status"]].head(20))


## 4. 학습 없이 입력 캐시 준비
1000×1000 크롭과 마스킹을 적용한 뒤 두 입력 모두 동일한 bilinear＋antialias로 512에 줄인다. 원본/마스크 해시를 검증한다. 캐시는 로컬 SSD에 약 5.1GiB(512 기준), RAM에 전체 적재하지 않는다.

In [ ]:
CACHE = WORK / f"cache_{IMG_SIZE}"
def run_logged(command, log_name):
    log_path = RESULTS / log_name
    with log_path.open("a", encoding="utf-8") as log:
        env = dict(os.environ, PYTHONUNBUFFERED="1")
        p = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, encoding="utf-8", errors="replace", env=env)
        try:
            for line in p.stdout:
                log.write(line); log.flush()
                print(line, end="")
            status = p.wait()
        except BaseException:
            p.terminate()
            p.wait()
            raise
    assert status == 0, f"실행 실패(code={status}). {log_path}를 확인해줘."

COMMON = [sys.executable, str(BUNDLE / "train_resnet18_threeway.py"),
          "--bundle", str(BUNDLE), "--originals", str(ORIGINALS),
          "--cache", str(CACHE), "--img-size", str(IMG_SIZE)]
run_logged(COMMON + ["--prepare-only"], "prepare.log")


## 5. 실제 학습 입력 6장 점검
위: 버전 1 원본 크롭, 아래: 버전 1.5/2 마스킹. 재활용 후보는 버전 2만 학습하지만 세 모델의 공통 진단 평가에는 모두 넣는다.

In [ ]:
import numpy as np
from PIL import Image, ImageDraw
selected_rows = [r for r in manifest if r["cohort"] != "held"]
raw = np.load(CACHE / "raw.npy", mmap_mode="r")
masked = np.load(CACHE / "masked.npy", mmap_mode="r")
pick = []
for cls in ("normal", "underfill_50"):
    for cohort in ("primary", "rescue"):
        ids = [i for i, r in enumerate(selected_rows) if r["class_name"] == cls and r["cohort"] == cohort]
        pick.append(ids[len(ids)//2])
pick += [min((i for i,r in enumerate(selected_rows) if r["cohort"]=="primary"), key=lambda i:float(selected_rows[i]["mask_area"])),
         min((i for i,r in enumerate(selected_rows) if r["cohort"]=="rescue"), key=lambda i:float(selected_rows[i]["mask_area"]))]
for i in pick:
    r = selected_rows[i]
    print(r["source"], r["cohort"], "mask area", round(float(r["mask_area"])*100, 1), "%")
    pair = Image.new("RGB", (600,300), "white")
    for j, a in enumerate((raw[i], masked[i])):
        pair.paste(Image.fromarray(a.transpose(1,2,0)).resize((300,300)), (j*300,0))
    display(pair)
del raw, masked


## 6. 본 학습 — 3버전 × 5 fold
한 round마다 로그와 재개 체크포인트를 Drive에 저장한다. 오류/OOM이면 자동으로 조건을 바꾸지 않고 멈춘다. batch나 크기를 바꾸려면 `RUN_NAME`도 새로 지정한다. 사전학습 가중치는 첫 실행에서 torchvision 공식 서버로부터 내려받는다.

In [ ]:
command = COMMON + ["--output", str(RESULTS), "--batch-size", str(BATCH_SIZE),
    "--rounds", str(ROUNDS), "--variants", *VARIANTS,
    "--folds", *map(str, FOLDS), "--seeds", *map(str, SEEDS)]
if FAST:
    command.append("--fast")
run_logged(command, "training.log")


## 7. 결과 표
`all_common`은 동일 원본 사진 집합, `primary`는 기존 통과 사진, `rescue`는 재활용 후보다. `pipeline_accepted`는 각 버전이 실제로 판정하는 집합이라 서로 분모가 다르다. 동일 분모 비교에는 `all_common`과 하위 집합을 사용한다. FP=정상 오탐, FN=언더필 미탐.

In [ ]:
summary = pd.read_csv(RESULTS / "summary.csv")
columns = ["variant", "seed", "cohort", "n", "accuracy", "balanced_accuracy", "auroc", "normal_recall", "underfill_recall", "fp", "fn", "completed_folds", "five_folds_complete"]
display(summary[columns].round(4))
display(pd.read_csv(RESULTS / "paired_comparison.csv").round(4))
display(pd.read_csv(RESULTS / "coverage.csv").round(4))
print("주의: 5 fold가 모두 끝났는지 확인. 부분 실행 점수는 완료한 같은 fold끼리만 비교.")


## 8. 공통 사진 성능 비교 그림
표의 검증된 값만 그린다. 오차 막대/유의성은 계산하지 않는다. 1개 seed의 시간 블록 예비 결과이지 새로운 출력 성능이나 인과적 증명이 아니다.

In [ ]:
import matplotlib.pyplot as plt
colors = {"v1":"#244b68", "v1_5":"#b88736", "v2":"#8a5279"}
groups = ["primary", "rescue", "all_common"]
fig, axes = plt.subplots(1, 3, figsize=(12, 4.5), sharey=True)
for ax, group in zip(axes, groups):
    d = summary[(summary["cohort"] == group) & (summary["seed"] == SEEDS[0])].set_index("variant")
    order = [v for v in VARIANTS if v in d.index]
    values = [100*d.loc[v,"balanced_accuracy"] for v in order]
    ax.bar(order, values, color=[colors[v] for v in order])
    ax.set_ylim(0, 105)
    ax.set_title(f"{group} (n={int(d.loc[order[0],'n'])})")
    ax.set_xlabel("Preprocessing variant")
    ax.spines[["top","right"]].set_visible(False)
    for x, val in enumerate(values):
        ax.text(x,val+1,f"{val:.1f}%",ha="center",fontsize=10)
axes[0].set_ylabel("Balanced accuracy (%)")
fig.suptitle(f"ResNet18 OOF comparison — seed {SEEDS[0]}, within-capture time blocks")
fig.tight_layout()
fig.savefig(RESULTS / "common_comparison.png", dpi=160)
plt.show()


## 9. 결과 묶음 저장
원본이나 캐시는 제외하고 점수·OOF·설정·로그를 ZIP으로 모은다. 모델 가중치와 재개용 체크포인트는 Drive 각 fold 폴더에 그대로 있다.

In [ ]:
result_zip = RESULTS.parent / f"{RUN_NAME}_results.zip"
with zipfile.ZipFile(result_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULTS.rglob("*")):
        if p.is_file() and p.suffix in (".csv", ".json", ".log", ".png"):
            z.write(p, p.relative_to(RESULTS).as_posix())
print("결과 폴더:", RESULTS)
print("공유할 결과 ZIP:", result_zip)
print("분석 포인트: 1↔1.5 마스킹 차이 / 1.5↔2 재활용 추가 차이 / rescue 미탐 감소와 primary 오탐 증가 여부")
print("검증 결과를 보고 설정을 다시 골랐다면 독립 테스트 성능으로 쓰지 않기.")


## 10. 런타임 반납(수동)
Drive의 결과 ZIP을 확인한 뒤 **런타임 → 연결 해제 및 런타임 삭제**를 눌러 GPU를 반납한다. 실행 도중 연결 해제로 로컬 캐시가 사라졌다면 1~6셀을 다시 실행해서 Drive 체크포인트에서 재개한다. 이 노트북은 아직 학습되지 않았으므로 실제 결과에 대한 결론은 실행 후 작성한다.